In [2]:

from pathlib import Path
import duckdb
import pandas as pd

## DuckDB setup

In [ ]:

REPO_ROOT = Path.cwd()

DATA_DIR = REPO_ROOT / "data"

FILES = {
    "backend_events": DATA_DIR / "backend_events.csv",
    "hubspot_deals": DATA_DIR / "hubspot_deals.csv",
    "hubspot_companies": DATA_DIR / "hubspot_companies.csv",
    "hubspot_contacts": DATA_DIR / "hubspot_contacts.csv",
}

DB_PATH = REPO_ROOT / "local.duckdb"  
con = duckdb.connect(str(DB_PATH))


for view_name, csv_path in FILES.items():
    csv_abs = csv_path.resolve().as_posix()
    con.execute(f"""
        CREATE OR REPLACE VIEW {view_name} AS
        SELECT *
        FROM read_csv_auto('{csv_abs}', HEADER=TRUE);
    """)

print("Views created:", ", ".join(FILES.keys()))
print("DB file:", DB_PATH)


Views created: backend_events, hubspot_deals, hubspot_companies, hubspot_contacts
DB file: /Users/vitalii/Desktop/unne/bi-technical-challenge/local.duckdb


## Question 1

In [49]:
sql = f"""
    SELECT        
    lifecycle_stage, count(*) AS n
    FROM hubspot_contacts
    right join hubspot_companies on hubspot_contacts.hubspot_company_id = hubspot_companies.company_id
     group by lifecycle_stage
    """

con.execute(sql).df()

,lifecycle_stage,n
0,opportunity,37
1,customer,137
2,lead,315


In [92]:
sql = f"""
    SELECT        
    company_id, lifecycle_stage, company_name, count(deal_id) as n_deals
    FROM hubspot_contacts
    right join hubspot_companies on hubspot_contacts.hubspot_company_id = hubspot_companies.company_id
    left join hubspot_deals on hubspot_deals.hubspot_company_id = hubspot_companies.company_id
    group by company_id, lifecycle_stage, company_name
    -- having n_deals == 0 and lifecycle_stage != 'lead'
    -- having lifecycle_stage == 'opportunity'
    order by n_deals desc
    
    """

con.execute(sql).df()

,company_id,lifecycle_stage,company_name,n_deals
0,1001191,customer,HealthBridge S.r.l.,16
1,1004319,customer,VascuTech S.r.l.,12
2,1004447,customer,WellPath d.o.o.,12
3,1002602,customer,NovaScan ApS,10
4,1004930,customer,ZenithBio Oy,10
...,...,...,...,...
248,1029888,lead,RiverHealth AG,0
249,1027471,lead,EagleBio S.A.,0
250,1047633,lead,KryptoHealth S.L.,0
251,1022561,lead,HorizonMed Inc,0


In [ ]:
# we want to count how many different lifecycle stages each company has among its contacts
sql = f"""
    SELECT        
    company_id, count(Distinct lifecycle_stage)
    FROM hubspot_companies c
    Left join hubspot_contacts p on p.hubspot_company_id = c.company_id 
    group by company_id
    order by 2 desc
    """

con.execute(sql).df()


,company_id,count(DISTINCT lifecycle_stage)
0,1019859,1
1,1012890,1
2,1037087,1
3,1042521,1
4,1024110,1
...,...,...
248,1000636,1
249,1014211,1
250,1040518,1
251,1014311,1


### Answer based on lifecicle stage

In [98]:
# since each company can have only one lifecycle stage, 
# we can define customers companies as those with lifecycle_stage = "customer"

sql = f"""
    SELECT        
    lifecycle_stage, 
    count(Distinct company_id) as n_companies, 
    count(company_id) as associated_contacts
    FROM hubspot_contacts
    right join hubspot_companies on hubspot_contacts.hubspot_company_id = hubspot_companies.company_id
    where lifecycle_stage = 'customer'
    group by lifecycle_stage
    """

con.execute(sql).df()

,lifecycle_stage,n_companies,associated_contacts
0,customer,28,137


In [120]:
# Playing with filters to find active deals

sql = f"""
    with stage_data as (
    SELECT        
    Distinct company_id, lifecycle_stage
    FROM hubspot_contacts p
    right join hubspot_companies c on p.hubspot_company_id = c.company_id
    )

    SELECT s.lifecycle_stage, d.*
    FROM hubspot_deals d
    join stage_data s on s.company_id = d.hubspot_company_id
    WHERE create_date is not null
    --and close_date is null
    and date_entered_closed_lost is not null
    limit 3
"""    
con.execute(sql).df()

,lifecycle_stage,deal_id,deal_name,pipeline,is_closed,is_closed_won,amount,close_date,create_date,hubspot_company_id,deal_type,currency,date_entered_pre_pitch,date_entered_pitching,date_entered_product_testing,date_entered_price_offering,date_entered_contract_negotiation,date_entered_closed_won,date_entered_closed_lost
0,lead,20019799,LaserMed ApS - Opportunity,Sales Pipeline,True,False,20000,2025-05-19 11:27:09,2025-04-20 11:27:09,1012160,newbusiness,EUR,2025-04-20 11:27:09,NaT,2025-05-03 11:27:09,NaT,2025-05-11 11:27:09,NaT,2025-05-12 11:27:09
1,lead,20023554,BrightPath Ltd - Opportunity,Sales Pipeline,True,False,10000,2025-08-21 20:52:13,2025-07-31 20:52:13,1015264,newbusiness,EUR,2025-07-31 20:52:13,NaT,2025-08-06 20:52:13,2025-08-16 20:52:13,2025-08-17 20:52:13,NaT,2025-08-20 20:52:13
2,lead,20019107,InfoSurg Oy - Opportunity,Sales Pipeline,True,False,10000,2025-10-31 10:37:54,2025-08-30 10:37:54,1011530,newbusiness,EUR,NaT,2025-08-30 10:37:54,2025-09-08 10:37:54,2025-09-12 10:37:54,2025-09-27 10:37:54,NaT,2025-10-18 10:37:54


In [113]:
# active deals 
sql = f"""
    with stage_data as (
    SELECT        
    Distinct company_id, lifecycle_stage
    FROM hubspot_companies c
    Left join hubspot_contacts p on p.hubspot_company_id = c.company_id 
    )

    SELECT s.lifecycle_stage, d.*
    FROM hubspot_deals d
    join stage_data s on s.company_id = d.hubspot_company_id
    and (close_date is not null or is_closed_won = true or date_entered_closed_won is not null)
    and date_entered_closed_lost is null
    order by hubspot_company_id, close_date desc
"""    
con.execute(sql).df()

,lifecycle_stage,deal_id,deal_name,pipeline,is_closed,is_closed_won,amount,close_date,create_date,hubspot_company_id,deal_type,currency,date_entered_pre_pitch,date_entered_pitching,date_entered_product_testing,date_entered_price_offering,date_entered_contract_negotiation,date_entered_closed_won,date_entered_closed_lost
0,customer,20000000,Acumed ApS - Annual License,Sales Pipeline,True,True,12000,2025-09-29 01:47:03,2025-08-25 01:47:03,1000000,newbusiness,EUR,NaT,2025-08-25 01:47:03,2025-09-15 01:47:03,NaT,2025-09-22 01:47:03,2025-09-28 01:47:03,NaT
1,customer,20000604,BioVance d.o.o. - Upsell,Sales Pipeline,True,True,10000,2026-01-06 17:05:48,2025-12-20 17:05:48,1000106,existing_business,EUR,2025-12-20 17:05:48,2025-12-23 17:05:48,2025-12-27 17:05:48,NaT,2026-01-01 17:05:48,2026-01-02 17:05:48,NaT
2,customer,20000420,BioVance d.o.o. - Annual License,Sales Pipeline,True,True,7500,2025-09-26 17:05:48,2025-05-24 17:05:48,1000106,newbusiness,EUR,2025-05-24 17:05:48,2025-08-14 17:05:48,2025-08-25 17:05:48,2025-09-14 17:05:48,NaT,2025-09-21 17:05:48,NaT
3,customer,20000884,ClearPath S.A. - Annual License,Sales Pipeline,True,True,8000,2024-09-17 23:01:51,2024-05-06 23:01:51,1000379,newbusiness,EUR,NaT,2024-05-06 23:01:51,2024-05-24 23:01:51,NaT,2024-06-02 23:01:51,2024-06-06 23:01:51,NaT
4,customer,20001030,DentalPro GmbH - Annual License,Sales Pipeline,True,True,12000,2025-09-11 21:47:57,2025-05-29 21:47:57,1000486,newbusiness,EUR,2025-05-29 21:47:57,NaT,2025-06-24 21:47:57,2025-07-25 21:47:57,2025-08-24 21:47:57,2025-09-09 21:47:57,NaT
5,customer,20001570,EvoMed SAS - Annual License,Sales Pipeline,True,True,20000,2025-07-06 19:20:40,2025-06-03 19:20:40,1000636,newbusiness,EUR,2025-06-03 19:20:40,NaT,2025-06-16 19:20:40,2025-06-17 19:20:40,NaT,2025-06-29 19:20:40,NaT
6,customer,20001923,FlexiSurg d.o.o. - Annual License,Sales Pipeline,True,True,18000,2025-01-31 20:31:59,2024-11-29 20:31:59,1000807,newbusiness,EUR,NaT,2024-11-29 20:31:59,2024-12-06 20:31:59,2024-12-29 20:31:59,NaT,2025-01-22 20:31:59,NaT
7,customer,20002740,HealthBridge S.r.l. - Upsell,Sales Pipeline,True,True,8000,2026-02-03 08:11:54,2025-12-27 08:11:54,1001191,existing_business,EUR,2025-12-27 08:11:54,2026-01-03 08:11:54,2026-01-09 08:11:54,NaT,2026-01-22 08:11:54,2026-01-31 08:11:54,NaT
8,customer,20002417,HealthBridge S.r.l. - Annual License,Sales Pipeline,True,True,5000,2025-07-04 08:11:54,2025-02-13 08:11:54,1001191,newbusiness,EUR,2025-02-13 08:11:54,2025-04-30 08:11:54,2025-05-29 08:11:54,NaT,NaT,2025-06-21 08:11:54,NaT
9,customer,20003220,InnoCare Oy - Annual License,Sales Pipeline,True,True,5000,2025-04-11 19:25:10,2024-11-30 19:25:10,1001477,newbusiness,EUR,2024-11-30 19:25:10,2025-02-02 19:25:10,NaT,2025-02-21 19:25:10,2025-03-03 19:25:10,2025-03-13 19:25:10,NaT


### Answer based on active deals

In [114]:
sql = f"""
    with stage_data as (
    SELECT        
    Distinct company_id, lifecycle_stage
    FROM hubspot_companies c
    Left join hubspot_contacts p on p.hubspot_company_id = c.company_id 
    )

    SELECT s.lifecycle_stage, d.hubspot_company_id, count(*) as n_deals
    FROM hubspot_deals d
    join stage_data s on s.company_id = d.hubspot_company_id
    and (close_date is not null or is_closed_won = true or date_entered_closed_won is not null)
    and date_entered_closed_lost is null
    group by s.lifecycle_stage, d.hubspot_company_id
"""    
con.execute(sql).df()

,lifecycle_stage,hubspot_company_id,n_deals
0,customer,1002316,1
1,customer,1002758,2
2,customer,1002045,1
3,customer,1004675,1
4,customer,1002602,2
5,customer,1000486,1
6,customer,1005485,1
7,customer,1003847,1
8,customer,1003517,1
9,customer,1004930,2


Investigation 25 customers with active deals VS. 28 companies flagged as customers

In [116]:
sql = f"""
    with marked_customers as (
        SELECT        
        lifecycle_stage, 
        company_id , 
        count(company_id) as associated_contacts
        FROM hubspot_contacts
        right join hubspot_companies on hubspot_contacts.hubspot_company_id = hubspot_companies.company_id
        where lifecycle_stage = 'customer'
        group by lifecycle_stage, company_id
    ),
    stage_data as (
        SELECT        
        Distinct company_id, lifecycle_stage
        FROM hubspot_companies c
        Left join hubspot_contacts p on p.hubspot_company_id = c.company_id 
    ),
    orgs_with_deals as (
        SELECT s.lifecycle_stage, d.hubspot_company_id, count(*) as n_deals
        FROM hubspot_deals d
        join stage_data s on s.company_id = d.hubspot_company_id
        and (close_date is not null or is_closed_won = true or date_entered_closed_won is not null)
        and date_entered_closed_lost is null
        group by s.lifecycle_stage, d.hubspot_company_id
        )
    SELECT *
    from marked_customers mc
    WHERE NOT EXISTS (
        SELECT 1 FROM orgs_with_deals d WHERE d.hubspot_company_id = mc.company_id
);
"""    
con.execute(sql).df()


,lifecycle_stage,company_id,associated_contacts
0,customer,1001848,3
1,customer,1000994,6


In [118]:
sql = f"""
    SELECT *    
    from hubspot_deals d
    where hubspot_company_id in (1001848, 1000994)
"""    
con.execute(sql).df()


,deal_id,deal_name,pipeline,is_closed,is_closed_won,amount,close_date,create_date,hubspot_company_id,deal_type,currency,date_entered_pre_pitch,date_entered_pitching,date_entered_product_testing,date_entered_price_offering,date_entered_contract_negotiation,date_entered_closed_won,date_entered_closed_lost
0,20002045,GenomaTech AG - Annual License,Sales Pipeline,True,False,8000,2025-06-17 10:32:16,2024-12-09 10:32:16,1000994,newbusiness,EUR,2024-12-09 10:32:16,2024-12-13 10:32:16,2024-12-22 10:32:16,2025-01-07 10:32:16,2025-01-24 10:32:16,2025-02-07 10:32:16,2025-06-17 10:32:16
1,20003652,KineMed S.L. - Annual License,Sales Pipeline,True,False,10000,2026-02-04 00:00:00,2025-05-29 19:17:59,1001848,newbusiness,EUR,2025-05-29 19:17:59,2025-07-07 19:17:59,2025-07-17 19:17:59,2025-08-10 19:17:59,2025-08-26 19:17:59,2025-09-30 19:17:59,2026-02-04 00:00:00
2,20003917,KineMed S.L. - Expansion,Sales Pipeline,False,False,10000,NaT,2025-12-24 16:59:33,1001848,existing_business,EUR,2025-12-24 16:59:33,2026-02-04 16:59:33,2026-02-07 16:59:33,2026-03-31 16:59:33,NaT,NaT,NaT


We should not include these companies (1001848, 1000994) into customers because their deals either lost or not yet prolonged.

### Final answer

-- How many customers do we have **today**?

-- **Today we have 26 customers.**



## Question 2

In [153]:
sql = f"""
    SELECT currency, 
    'won and havent lost' as deal_status,
    sum(amount) as total_amount, 
    count(*) as n_deals
    from hubspot_deals d
    where is_closed_won = true
    group by currency, deal_status
    UNION ALL
    
    SELECT currency, 
    'won initially' as deal_status,
    sum(amount) as total_amount, 
    count(*) as n_deals
    from hubspot_deals d
    where date_entered_closed_won is not null
    group by currency, deal_status
"""
con.execute(sql).df()

,currency,deal_status,total_amount,n_deals
0,EUR,won and havent lost,402000.0,31
1,EUR,won initially,420000.0,33


As we already explored, we have 2 deals, that became lost after a few active moths.

For ACV we should include deals that "won initially" assuming that the payment was done. 

In [ ]:
# Assumtion: Average Contract Value is calculated as the total amount of all initially closed as won deals divided by the number of them.
sql = f"""
    SELECT 
    case when deal_name like '%Expansion%' then 'Expansion'
         when deal_name like '%Upsell%' then 'Upsell'
         when deal_name like '%Annual License%' then 'Annual License'
         else 'Other' end as contract_type,
    currency,
    sum(amount) / count(*) as ACV,
    count(*) as n_deals,
    sum(amount) as total_amount
    from hubspot_deals d
    -- where is_closed_won = true
    where date_entered_closed_won is not null
    group by 1,2
    order by n_deals desc
"""
con.execute(sql).df()

,contract_type,currency,ACV,n_deals,total_amount
0,Annual License,EUR,13214.285714,28,370000.0
1,Upsell,EUR,10000.000000,5,50000.0


### Final answer

In [151]:

sql = f"""
with summary_table as (
    SELECT 
    case when deal_name like '%Expansion%' then 'Expansion'
         when deal_name like '%Upsell%' then 'Upsell'
         when deal_name like '%Annual License%' then 'Annual License'
         else 'Other' end as contract_type,
    currency,
    sum(amount) / count(*) as ACV,
    count(*) as n_deals,
    sum(amount) as total_amount
    from hubspot_deals d
    -- where is_closed_won = true
    where date_entered_closed_won is not null
    group by 1,2
    order by n_deals desc
  )
SELECT contract_type,
    n_deals,
    total_amount,
    Round(ACV,2) as contract_type_ACV_euros,
    Round(SUM(total_amount) OVER () / SUM(n_deals) OVER (), 2) AS Total_ACV_euros
FROM summary_table
"""
con.execute(sql).df()

,contract_type,n_deals,total_amount,contract_type_ACV_euros,Total_ACV_euros
0,Annual License,28,370000.0,13214.29,12727.27
1,Upsell,5,50000.0,10000.00,12727.27


## Question 3

In [154]:
sql = f"""
    SELECT        
    *
    FROM backend_events
    limit 5
    """

con.execute(sql).df()

,event_id,event_name,event_timestamp,user_id,organization_id,event_properties
0,907ca6c5-a6f7-4b9d-bb00-e1ac41296202,UserCreated,2024-07-03 09:42:25,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-235415c6b85e"", ""email"": ""thomas.hoffmann@alphadevice.eu""}, ""organization"": {""id"": ""1dedfc50-1e5b-4624-8fd1-c689de270bbc""}}"
1,4eb6b45f-1a1a-4b5c-aab0-7b83148fd8a7,OrganizationCreated,2024-07-03 09:42:25,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""organization"": {""id"": ""1dedfc50-1e5b-4624-8fd1-c689de270bbc""}, ""user"": {""id"": ""146dda75-703c-462b-9038-235415c6b85e""}}"
2,e5a7e460-7607-4b4b-8bd1-bdee53e9baaa,SearchUpdated,2024-07-03 13:33:52,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-235415c6b85e""}, ""organization"": {""id"": ""1dedfc50-1e5b-4624-8fd1-c689de270bbc""}, ""search"": {""id"": ""1bfcde4a-218f-4c53-a889-59f035b6f493""}}"
3,14b216e9-dc98-4762-a395-8fcb54325bfe,SearchExecuted,2024-07-05 18:27:49,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-235415c6b85e""}, ""organization"": {""id"": ""1dedfc50-1e5b-4624-8fd1-c689de270bbc""}, ""search"": {""id"": ""ee0afe7f-4286-4ac9-888b-bae1ecf0233a""}}"
4,0cb57b2f-8caf-4228-b98a-07e923393f20,SearchResultAppraised,2024-07-05 23:42:38,146dda75-703c-462b-9038-235415c6b85e,1dedfc50-1e5b-4624-8fd1-c689de270bbc,"{""user"": {""id"": ""146dda75-703c-462b-9038-235415c6b85e""}, ""organization"": {""id"": ""1dedfc50-1e5b-4624-8fd1-c689de270bbc""}, ""search"": {""id"": ""765ce420-5291-4192-a0b4-288d0204ba6b""}, ""search_result"": {""id"": ""907eb8e1-31dc-4ef1-af43-16e67be42aa1""}}"


In [ ]:
sql = f"""
    SELECT     
    json_valid(event_properties) AS is_valid,   
    json_structure(event_properties) AS event_properties_structured
    FROM backend_events
    limit 5
    """
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 240)
con.execute(sql).df()

,is_valid,event_properties_structured
0,True,"{""user"":{""id"":""VARCHAR"",""email"":""VARCHAR""},""organization"":{""id"":""VARCHAR""}}"
1,True,"{""organization"":{""id"":""VARCHAR""},""user"":{""id"":""VARCHAR""}}"
2,True,"{""user"":{""id"":""VARCHAR""},""organization"":{""id"":""VARCHAR""},""search"":{""id"":""VARCHAR""}}"
3,True,"{""user"":{""id"":""VARCHAR""},""organization"":{""id"":""VARCHAR""},""search"":{""id"":""VARCHAR""}}"
4,True,"{""user"":{""id"":""VARCHAR""},""organization"":{""id"":""VARCHAR""},""search"":{""id"":""VARCHAR""},""search_result"":{""id"":""VARCHAR""}}"


In [155]:
sql = """
WITH base AS (
  SELECT
    event_id,
    CASE
      WHEN event_properties IS NULL OR TRIM(event_properties) = '' THEN NULL
      WHEN json_valid(event_properties) THEN json(event_properties)
      ELSE NULL
    END AS ep
  FROM backend_events
),
valid AS (
  SELECT COUNT(*) AS n FROM base WHERE ep IS NOT NULL
),
keys AS (
  SELECT
    b.event_id,
    t.key AS key
  FROM base b,
       UNNEST(json_keys(b.ep)) AS t(key)
  WHERE b.ep IS NOT NULL
)
SELECT
  key,
  COUNT(*) AS rows_with_key,
  ROUND(100.0 * COUNT(*) / (SELECT n FROM valid), 2) AS pct_of_valid_rows
FROM keys
GROUP BY 1
ORDER BY rows_with_key DESC
LIMIT 200;
"""

con.execute(sql).df()

,key,rows_with_key,pct_of_valid_rows
0,user,39206,100.00
1,organization,39206,100.00
2,search,25601,65.30
3,search_result,10228,26.09
4,label,1210,3.09


In [ ]:

sql = f"""
    SELECT     
    count(DISTINCT event_properties->'user'->>'id') AS n_users_with_email
    FROM backend_events
    WHERE event_properties->'user'->>'email' IS NOT NULL
    limit 5
    """

con.execute(sql).df()

,n_events_with_user_email
0,150


In [181]:

sql = """
    SELECT     
    count(DISTINCT user_id) AS n_users,
    SUM(CASE WHEN event_properties->'user'->>'email' IS NOT NULL THEN 1 ELSE 0 END) AS n_events_with_email
    FROM backend_events
    where event_timestamp >= '2026-01-01' 
"""

con.execute(sql).df()

,n_users,n_events_with_email
0,131,2.0


In [180]:

sql = """
    SELECT     
    date_part('yearweek', event_timestamp) as yearweek,
    date_part('week', event_timestamp) as week,
    count(DISTINCT user_id) AS n_users,
    count(DISTINCT CASE WHEN event_properties->'user'->>'email' IS NOT NULL THEN user_id END) AS n_users_with_email
    FROM backend_events
    where event_timestamp >= '2026-01-01' 
    AND user_id IS NOT NULL
    AND event_name NOT IN (
      'TokenGenerated',
      'UserCreated', 'UserUpdated',
      'OrganizationCreated', 'OrganizationUpdated')
    group by 1,2
    order by yearweek
"""

con.execute(sql).df()

,yearweek,week,n_users,n_users_with_email
0,202601,1,97,0
1,202602,2,110,0
2,202603,3,107,0
3,202604,4,110,0
4,202605,5,112,0
5,202606,6,108,0


### Final answer

In [ ]:
sql = """
with backend_emails as (
    SELECT     
    event_properties->'user'->>'email' as email_present,
    user_id
    FROM backend_events
    where event_name = 'UserCreated'
),
customer_users as (
Select user_id, contact_id from backend_emails
join hubspot_contacts on hubspot_contacts.email = backend_emails.email_present
    and lifecycle_stage = 'customer'
),
active_events AS (
  SELECT
    user_id,
    date_trunc('week', event_timestamp) AS week_start
  FROM backend_events
  join customer_users using (user_id)
  WHERE event_timestamp >= TIMESTAMP '2026-01-01'
    AND user_id IS NOT NULL
    AND event_name NOT IN (
      'TokenGenerated',
      'UserCreated', 'UserUpdated',
      'OrganizationCreated', 'OrganizationUpdated'
    )
),
active_users AS (
  SELECT DISTINCT user_id, week_start
  FROM active_events
),
first_week AS (
  SELECT user_id, MIN(week_start) AS cohort_week
  FROM active_users
  GROUP BY 1
),
cohort_activity AS (
  SELECT
    f.cohort_week,
    a.week_start,
    datediff('week', f.cohort_week, a.week_start) AS week_index,
    a.user_id
  FROM first_week f
  JOIN active_users a USING (user_id)
)
SELECT
  date_part('yearweek', cohort_week) AS cohort_yearweek,
  week_index,
  COUNT(DISTINCT user_id) AS n_users_active,
  MAX(CASE WHEN week_index = 0 THEN COUNT(DISTINCT user_id) END) OVER (PARTITION BY cohort_week) AS cohort_size,
  ROUND(
    100.0 * COUNT(DISTINCT user_id)
    / NULLIF(MAX(CASE WHEN week_index = 0 THEN COUNT(DISTINCT user_id) END) OVER (PARTITION BY cohort_week), 0),
    2
  ) AS retention_pct
FROM cohort_activity
WHERE week_index >= 0
GROUP BY 1,2, cohort_week
ORDER BY cohort_week, week_index;
"""

con.execute(sql).df()

,cohort_yearweek,week_index,n_users_active,cohort_size,retention_pct
0,202601,0,91,91,100.00
1,202601,1,87,91,95.60
2,202601,2,81,91,89.01
3,202601,3,85,91,93.41
4,202601,4,82,91,90.11
5,202601,5,83,91,91.21
6,202602,0,18,18,100.00
7,202602,1,15,18,83.33
8,202602,2,13,18,72.22
9,202602,3,14,18,77.78


## Additional insight

In [199]:
sql = f"""
    select job_title, count(*) as n
    from hubspot_contacts
    group by job_title
    order by n desc
    """

con.execute(sql).df()

,job_title,n
0,Technical Documentation Specialist,40
1,Regulatory Specialist,35
2,Compliance Officer,33
3,Regulatory Affairs Associate,33
4,CEO,32
5,Director Regulatory Affairs,30
6,Senior Regulatory Consultant,29
7,Clinical Affairs Manager,28
8,Product Manager,27
9,QA/RA Engineer,27


In [ ]:

sql = r"""
WITH base AS (
  SELECT
    contact_id,
    COALESCE(NULLIF(TRIM(job_title), ''), 'Unknown') AS job_title,
    LOWER(COALESCE(NULLIF(TRIM(lifecycle_stage), ''), 'lead')) AS lifecycle_stage
  FROM hubspot_contacts
  WHERE LOWER(COALESCE(NULLIF(TRIM(lifecycle_stage), ''), 'lead')) IN ('lead', 'opportunity', 'customer')
),
role_grouped AS (
  SELECT
    contact_id,
    lifecycle_stage,
    CASE
      -- Group 1: Executive / Leadership
      WHEN regexp_matches(LOWER(job_title), '(ceo|chief executive|coo|chief operating|cto|chief technology|cfo|chief financial|founder|president|vp|vice president|head|director)')
        THEN 'Leadership'

      -- Group 2: Regulatory / Clinical / Compliance
      WHEN regexp_matches(LOWER(job_title), '(regulatory|compliance|clinical|affairs)')
        THEN 'Regulatory/Clinical'

      -- Group 3: Quality / Risk / QA/RA / Post-market
      WHEN regexp_matches(LOWER(job_title), '(quality|qa|ra|risk|post[- ]?market|surveillance)')
        THEN 'Quality/Risk'

      -- Group 4: Product / Engineering / Documentation / Everything else
      ELSE 'Product/Tech/Other'
    END AS job_group
  FROM base
),
agg AS (
  SELECT
    lifecycle_stage,
    job_group,
    COUNT(DISTINCT contact_id) AS n_contacts
  FROM role_grouped
  GROUP BY 1,2
),
stage_totals AS (
  SELECT
    lifecycle_stage,
    SUM(n_contacts) AS stage_total
  FROM agg
  GROUP BY 1
)
SELECT
  a.lifecycle_stage,
  a.job_group,
  a.n_contacts,
  t.stage_total,
  (1.0 * a.n_contacts) / NULLIF(t.stage_total, 0) AS pct_within_stage
FROM agg a
JOIN stage_totals t USING (lifecycle_stage)
ORDER BY
  CASE a.lifecycle_stage
    WHEN 'lead' THEN 1
    WHEN 'opportunity' THEN 2
    WHEN 'customer' THEN 3
    ELSE 9
  END,
  a.job_group;
"""
df = con.execute(sql).df()
df

,lifecycle_stage,job_group,n_contacts,stage_total,pct_within_stage
0,lead,Leadership,104,315.0,0.330159
1,lead,Product/Tech/Other,43,315.0,0.136508
2,lead,Quality/Risk,54,315.0,0.171429
3,lead,Regulatory/Clinical,114,315.0,0.361905
4,opportunity,Leadership,14,37.0,0.378378
5,opportunity,Product/Tech/Other,1,37.0,0.027027
6,opportunity,Quality/Risk,6,37.0,0.162162
7,opportunity,Regulatory/Clinical,16,37.0,0.432432
8,customer,Leadership,50,137.0,0.364964
9,customer,Product/Tech/Other,23,137.0,0.167883


In [207]:

import altair as alt

stage_order = ["lead", "opportunity", "customer"]
group_order = ["Leadership", "Regulatory/Clinical", "Quality/Risk", "Product/Tech/Other"]

plot_df = df.copy()
plot_df["lifecycle_stage"] = pd.Categorical(plot_df["lifecycle_stage"], categories=stage_order, ordered=True)
plot_df["job_group"] = pd.Categorical(plot_df["job_group"], categories=group_order, ordered=True)

# Format percent for labels/tooltips
plot_df["pct_label"] = (plot_df["pct_within_stage"] * 100).round(1).astype(str) + "%"

chart = (
    alt.Chart(plot_df)
    .mark_bar()
    .encode(
        x=alt.X("lifecycle_stage:N", sort=stage_order, title="Lifecycle stage"),
        y=alt.Y("pct_within_stage:Q", stack="normalize", title="Share of contacts (normalized)"),
        color=alt.Color("job_group:N", sort=group_order, title="Job group"),
        tooltip=[
            alt.Tooltip("lifecycle_stage:N", title="Stage"),
            alt.Tooltip("job_group:N", title="Job group"),
            alt.Tooltip("n_contacts:Q", title="Contacts"),
            alt.Tooltip("stage_total:Q", title="Stage total"),
            alt.Tooltip("pct_within_stage:Q", title="Share", format=".1%")
        ],
        order=alt.Order("job_group:N", sort="ascending"),
    )
    .properties(
        width=420,
        height=420,
        title="Job groups by lifecycle stage (normalized stacked bars)"
    )
)


(chart).configure_axis(
    labelFontSize=12,
    titleFontSize=12
).configure_legend(
    titleFontSize=12,
    labelFontSize=12
).configure_title(
    fontSize=14
)

alt.Chart(...)

In [209]:

from pathlib import Path


final_chart = chart

out_dir = Path("outputs")
docs_dir = Path("docs")  # GitHub Pages commonly serves from /docs
out_dir.mkdir(exist_ok=True)
docs_dir.mkdir(exist_ok=True)

png_path = out_dir / "job_groups_by_stage.png"
html_path = docs_dir / "job_groups_by_stage.html"

# Interactive HTML (keeps hover tooltips)
final_chart.save(str(html_path))

# Static PNG (for README)
final_chart.save(str(png_path), scale_factor=2)

print("Saved:")
print(" - PNG  :", png_path)
print(" - HTML :", html_path)

Saved:
 - PNG  : outputs/job_groups_by_stage.png
 - HTML : docs/job_groups_by_stage.html


In [ ]:
# con.close()